**Objetivo:** saber qué hay, cuánto hay y si hay "nulo simples", para decidir qué propuesta de kpi buscar

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
from src.carga import cargar_bronze

dfs = cargar_bronze()
dfs.keys()   # agencias, empresas, ocupaciones, reservaciones, segmentos_mercados

 Al momento sé que 
- `agencias`, el de agencias de viaje;
- `empresas` es el catálogo de hoteles del grupo; 
- `ocupaciones` contiene ingresos, ADR y cuartos noche a lo largo del tiempo;
- `segmentos_mercados`, el de mercados;
- `reservaciones`, puede tener la información más importante para KPIs junto a ocupaciones

In [ ]:
inventario = pd.DataFrame({
    nombre: {
        "filas": len(df),
        "columnas": df.shape[1],
        "numericas": df.select_dtypes("number").shape[1],
        "texto": df.select_dtypes(["object", "string"]).shape[1],
        "fechas": df.select_dtypes("datetime").shape[1],
        "duplicados_exactos": int(df.duplicated().sum()),
    }
    for nombre, df in dfs.items()
}).T.sort_values("filas", ascending=False)
inventario

Ojo, por las pruebas indivuales anteriores que hice, sé que hay agencias duplicadas, pero, al tener la columna id no las está detectando

In [ ]:
def perfil(df):
    """Tipo, % de nulos, valores únicos y un ejemplo por columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos_%": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(),
        "ejemplo": df.iloc[0] if len(df) else None,
    })

In [ ]:
for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    display(perfil(df))

# Preguntas y observaciones que me surgen:
### Para reservas:
- ¿Qué es el estado_huésped en reservaciones? Por el resultado observado, no parece ser algo de finalizado el servicio de hotelería, sino, parece más un código de zona. ¿EMX? estado mexico ?
- Debería estandarizar el formato de la fecha_hoy o me fío de la fecha de reserva? parece ser la misma pero formateada
### Para ocupaciones:
- ¿Por qué hay tres columnas de que parecen registrar lo mismo: menores, juniors e infantiles? 
- ¿Puedo calcular los ingresos_habitaciones al cruzarlo con tarifa toal de reserva?
- ¿Puedo recalcular ADR con las tablas que tengo?
- ¿Qué es trevpec? 
> Tanto para ocupaciones y reservas parece que solo se ven involucradas 5 empresas (al hacer el recuento único de ids en esa columna)

In [ ]:
dfs["ocupaciones"].describe().T

In [ ]:
dfs["reservaciones"].describe().T

In [ ]:
display(dfs["agencias"].head(3))
display(dfs["segmentos_mercados"])

Me centraré en un nuevo notebook para hacer algunos joins y ver si puedo relacionar ocupaciones, reservaciones con los respectivos catálogos